# 🍀 YOTSUBA FINDER — 四つ葉検出モデルの学習

このノートブックは、上から順にセルを実行するだけで、次のことを行います。

1. GPU の確認とライブラリの準備
2. Google ドライブから学習データ（`merged_core.zip`・`merged_all.zip`・`merged_hardneg.zip` のどれか）を読み込む
3. YOLO（nano サイズ）で学習する
4. クラスごとの成績と混同行列を確認する
5. valid の画像 10 枚で、検出結果を目で確認する
6. アプリで使うために ONNX 形式（`yotsuba_<RUN_TAG>.onnx`）に書き出す

学習し直したときは、セクション 7-3 で前のモデルと同じ valid で比べられます。

**クラス**: `0 = four_leaf`（四つ葉）、`1 = three_leaf`（三つ葉）

> 実行前に「ランタイム」→「ランタイムのタイプを変更」で **GPU（T4 など）** を選んでください。

## 1. GPU の確認

GPU が使えるかを確かめます。表に `Tesla T4` などの名前が出れば OK です。
`command not found` や `NVIDIA-SMI has failed` と出たら、GPU が選ばれていません。

In [ ]:
# GPU の名前とメモリを表示する
!nvidia-smi

## 2. ライブラリのインストール

- **ultralytics**: YOLO の学習・評価・書き出しをまとめて行えるライブラリ
- **onnxruntime**: 書き出した ONNX モデルを試しに動かして、出力の形を確かめるために使う

In [ ]:
# -q は、インストール中の表示を少なくするオプション
!pip install -q ultralytics onnxruntime

import ultralytics
ultralytics.checks()  # バージョン・GPU・メモリなどの環境を表示する

## 3. Google ドライブの接続と、保存場所の設定

ドライブを接続すると、許可を求める画面が出ます。自分のアカウントを選んで許可してください。

| 置き場所 | 中身 |
|---|---|
| `MyDrive/yotsuba/merged_core.zip`<br>`MyDrive/yotsuba/merged_all.zip`<br>`MyDrive/yotsuba/merged_hardneg.zip` | 学習データ（**使うものを事前にアップロードしておく**） |
| `MyDrive/yotsuba/runs/<RUN_NAME>/` | 今回の学習の結果（重み・グラフ）。接続が切れても残る |
| `MyDrive/yotsuba/yotsuba_<RUN_TAG>.onnx` | 今回アプリ用に書き出すモデル |

**`DATASET_ZIP`** で、使う学習データを選びます。選んだ種類（core / all / hardneg）は、実行名と ONNX の名前に入ります。

**`START_FROM`** で、学習の始め方を選びます。
- `'pretrained'` … 事前学習済みの `yolo11n.pt` から、80 周学習する（これまでと同じ）
- `'v2'` … v2 の `best.pt`（`V2_BEST`）から続けて、30 周だけ、小さめの学習率で学習する（`merged_hardneg.zip`・`merged_hardneg2.zip` 用）

**学習し直すときは `RUN_TAG` を変えてください。** 実行名と ONNX の名前に入るので、前の学習結果や `yotsuba.onnx` を上書きしません。
`PREV_BEST` には、比べる相手（前のモデル）の `best.pt` を指定します。

In [ ]:
from pathlib import Path
from google.colab import drive

# ── 使う学習データ（どちらかを選ぶ）──
#   'merged_core.zip' … findclover・rectangle_annotation・hunting-four-leaf（ラベルが比較的ていねい）
#   'merged_all.zip'  … core ＋ clover-detection2（草むらの写真が多い。ラベルは粗め）
#   'merged_hardneg.zip' … core ＋ iNaturalist の「四つ葉がない」写真の窓（三つ葉を四つ葉と間違える癖を減らす。v3）
#   'merged_hardneg2.zip' … core ＋ iNaturalist の窓 ＋ core の窓（窓の偏りをなくした版。葉をまん中からずらし、四つ葉と三つ葉の数をそろえた）
#   'merged_hardneg2b.zip' … hardneg2 の作り直し（写真の外も灰色で埋める窓・iNaturalist の窓をすべて使う・灰色とくっきり具合の偏りをそろえた）
#   どれも valid は同じなので、成績をそのまま比べられる
DATASET_ZIP = 'merged_hardneg2b.zip'
DATASET_KIND = next(k for k in ('hardneg', 'core', 'all') if k in DATASET_ZIP)  # 実行名と ONNX の名前に入れる

# ── 学習の始め方 ──
#   'pretrained' … 事前学習済みの yolo11n.pt から、80 周学習する（これまでと同じ）
#   'v2'         … v2 の best.pt（V2_BEST）から続けて、30 周・小さめの学習率で学習する（merged_hardneg 用）
START_FROM = 'v2'

# Google ドライブを /content/drive に接続する
drive.mount('/content/drive')

# ── 場所の設定（ここを変えれば保存先を変えられる）──
DRIVE_DIR = Path('/content/drive/MyDrive/yotsuba')  # ドライブ上の作業フォルダ
ZIP_PATH = DRIVE_DIR / DATASET_ZIP                  # 学習データの zip
RUNS_DIR = DRIVE_DIR / 'runs'                       # 学習結果の保存先
LOCAL_DATA_DIR = Path('/content/datasets')          # zip の展開先（Colab のローカル）

# ── 学習の設定 ──
MODEL_NAME = 'yolo11n.pt'  # 事前学習済みの nano モデル（小さくて速い）
IMGSZ = 960                # 学習・推論で使う画像サイズ（小さい葉を見つけやすくする）
CLASS_NAMES = {0: 'four_leaf', 1: 'three_leaf'}

# ── 今回の学習の名前（前の結果を上書きしないように、学習し直すたびに変える）──
RUN_TAG = 'hardneg2b'                  # 今回の変更を表す短い名前
RUN_NAME = f'yolo11n_960_{DATASET_KIND}_{RUN_TAG}'    # 学習結果のフォルダ名（runs/ の下）
ONNX_NAME = f'yotsuba_{DATASET_KIND}_{RUN_TAG}.onnx'  # 書き出す ONNX の名前（前の yotsuba.onnx は上書きしない）

# ── v2（今アプリで使っているモデル）の best.pt ──
# v2 は merged_core・RUN_TAG='scale09' で学習したもの。ドライブの場所が違えば直す
V2_BEST = RUNS_DIR / 'yolo11n_960_core_scale09' / 'weights' / 'best.pt'

# ── 比べる相手（前のモデル）── v2 と比べる
PREV_BEST = V2_BEST

RUNS_DIR.mkdir(parents=True, exist_ok=True)

def latest_run():
    """runs/ の中で、今回の実行名（RUN_NAME）で始まり、途中経過（weights/last.pt）があるフォルダのうち、
    いちばん新しく保存されたものを返す（なければ None）。
    接続が切れたあとの再開（セクション 5-2）や、結果の表示（セクション 6）で使う。
    （last.pt のないフォルダは、学習を始めてすぐ止まったものなので選ばない）"""
    runs = [p for p in RUNS_DIR.iterdir() if p.name.startswith(RUN_NAME) and (p / 'weights' / 'last.pt').exists()]
    return max(runs, key=lambda p: (p / 'weights' / 'last.pt').stat().st_mtime) if runs else None

print(f'今回の実行名: {RUN_NAME} / 書き出す ONNX: {ONNX_NAME} / 始め方: {START_FROM}')
if START_FROM == 'v2' and not V2_BEST.exists():
    raise FileNotFoundError(f'❌ v2 の best.pt（{V2_BEST}）がありません。V2_BEST を直すか、START_FROM を \'pretrained\' にしてください。')
if PREV_BEST.exists():
    print(f'✅ 比べる相手（前のモデル）: {PREV_BEST}')
else:
    print(f'⚠️ 前のモデル {PREV_BEST} が見つかりません。比べる表（セクション 7-3）は飛ばされます。PREV_BEST を直してください。')

# zip が置かれているかを確かめる
if ZIP_PATH.exists():
    print(f'✅ 学習データが見つかりました: {ZIP_PATH}（{ZIP_PATH.stat().st_size / 1e6:.0f} MB）')
else:
    raise FileNotFoundError(f'❌ {ZIP_PATH} がありません。ドライブの MyDrive/yotsuba/ に {DATASET_ZIP} を置いてください。')

## 4. 学習データの展開と、Colab 用の data.yaml 作成

ドライブから直接読むと遅いので、zip を Colab のローカル（`/content/datasets`）にコピーして展開します。

zip の中の `data.yaml` には `path:`（データの場所）が書かれていません。
そこで、展開先を `path:` に書き込んだコピー **`data_colab.yaml`** を作り、学習ではこちらを使います。

In [ ]:
import shutil, zipfile

# 前回の展開が残っていたら消してから、展開し直す
shutil.rmtree(LOCAL_DATA_DIR, ignore_errors=True)
LOCAL_DATA_DIR.mkdir(parents=True)

# ドライブの zip をローカルにコピーしてから展開する（直接展開するより速い）
local_zip = Path('/content') / DATASET_ZIP
shutil.copy(ZIP_PATH, local_zip)
with zipfile.ZipFile(local_zip) as z:
    z.extractall(LOCAL_DATA_DIR)

DATA_DIR = LOCAL_DATA_DIR / Path(DATASET_ZIP).stem  # zip の中は merged_core/・merged_all/・merged_hardneg/ のどれかのフォルダ

# path: を書き込んだ data_colab.yaml を作る
DATA_YAML = DATA_DIR / 'data_colab.yaml'
names = '\n'.join(f'  {i}: {n}' for i, n in CLASS_NAMES.items())
DATA_YAML.write_text(
    f'path: {DATA_DIR}\n'
    'train: train/images\n'
    'val: valid/images\n'
    f'nc: {len(CLASS_NAMES)}\n'
    f'names:\n{names}\n'
)
print(DATA_YAML.read_text())

In [ ]:
from collections import Counter

# 画像の枚数と、クラスごとの枠の数を数えて、データが正しく展開できたか確かめる
for split in ['train', 'valid']:
    images = list((DATA_DIR / split / 'images').iterdir())
    boxes = Counter()
    for label in (DATA_DIR / split / 'labels').glob('*.txt'):
        for line in label.read_text().splitlines():
            if line.strip():
                boxes[int(line.split()[0])] += 1
    detail = '、'.join(f'{CLASS_NAMES[c]} {boxes[c]} 枠' for c in CLASS_NAMES)
    print(f'{split:<6}: 画像 {len(images)} 枚 / {detail}')

**読み方**: `prepare_dataset.py` を実行したときに最後に出た表（merged_core か merged_all）と同じ数なら OK です。
`three_leaf` は `four_leaf` よりかなり少ないので、三つ葉の成績は低めに出やすいことを覚えておいてください。

## 5. 学習

`START_FROM = 'pretrained'` のときは、事前学習済みの `yolo11n.pt` から始めて、四つ葉・三つ葉を見分けられるように学習させます（下の表の設定）。
`START_FROM = 'v2'` のときの設定は、表の下の「v2 から続けて学習するとき」を見てください。

| 設定 | 値 | 意味 |
|---|---|---|
| `imgsz` | 960 | 画像を 960px に縮めて学習する。大きいほど小さい葉が見えやすいが、時間とメモリが増える |
| `epochs` | 80 | 学習データ全体を最大 80 周する |
| `batch` | 16 | 一度に 16 枚ずつ学習する |
| `patience` | 25 | 25 周続けて成績が上がらなければ、早めに打ち切る |
| `flipud` | 0.5 | 半分の確率で画像を上下反転する（上から撮った写真には上下がないため） |
| `degrees` | 15 | ±15° の範囲で画像を回転する |
| `scale` | **0.9** | 画像を 0.1〜1.9 倍の範囲で、ランダムに拡大・縮小する（既定は 0.5） |
| `translate` | **0.2** | 画像を縦横 ±20% の範囲で、ランダムにずらす（既定は 0.1） |
| `close_mosaic` | **15** | 最後の 15 周は、モザイク（4 枚の画像を 1 枚に貼り合わせる拡張）を止める（既定は 10） |
| `name` | `RUN_NAME` | 結果のフォルダ名。前の学習（`yolo11n_960`）を上書きしない |

そのほかの拡張（左右反転・明るさの変化・モザイクなど）は Ultralytics の既定のままです。

**v2 から続けて学習するとき（`START_FROM = 'v2'`）**

| 設定 | 値 | 意味 |
|---|---|---|
| 始める重み | `V2_BEST` | v2 の覚えたことを残したまま、足したデータ（三つ葉の窓）を学ばせる |
| `epochs` | 30 | 最初から学ぶわけではないので、短くする |
| `optimizer` | SGD | 学習率を自分で決めるため（`auto` だと学習率が自動で決まり、`lr0` が使われない） |
| `lr0` | 0.001 | 学習率（既定の 0.01 の 1/10）。大きいと、v2 の覚えたことが崩れやすい |
| `lrf` | 0.1 | 最後の周の学習率 = `lr0` × 0.1 |
| `warmup_epochs` | 1 | 学習率を、最初の 1 周でゆっくり上げる（既定は 3） |
| `patience` | 10 | 10 周続けて成績が上がらなければ、早めに打ち切る |
| `close_mosaic` | 5 | 最後の 5 周はモザイクを止める |

拡張（`flipud`・`degrees`・`scale`・`translate`）は、v2 と同じです。

### 5-1. 今回の変更点と理由

前のモデルには、**「大きく写った葉は四つ葉、小さく写った葉は三つ葉」と、葉の大きさで決めてしまう癖**がありそうでした。
学習データの四つ葉は大きく（アップで）写っているものが多く、三つ葉は群生の中で小さく写っているものが多いためです。
そこで、「大きさ」を手がかりにできないように、次の 3 つを変えました。

- **`scale=0.9`**：同じ葉を、とても小さくしたり大きくしたりして見せます。四つ葉も三つ葉もいろいろな大きさで出てくるので、「大きさ」ではなく「葉の数・形」で見分けるしかなくなります
- **`translate=0.2`**：葉の位置を大きくずらします。真ん中にある葉＝四つ葉、のような「場所の癖」も覚えにくくなります。画像の端で一部が切れた葉も増えます
- **`close_mosaic=15`**：モザイクは画像を縮めて貼り合わせるので、小さな葉が増えます。最後の 15 周はモザイクを止め、ふつうの写真に近い見え方で仕上げます

これで本当に良くなったかは、学習してみないと分かりません。**セクション 7-3 で、前のモデルと同じ valid で比べて確かめます。**

⏱ 時間の目安（T4）: merged_core で **2〜3 時間**、merged_all で **4〜5 時間**ほど（画像の数からの見込み。早めに打ち切られると短くなります）。
merged_hardneg2b を v2 から 30 周なら、**2〜3 時間**ほど（train 4,695 枚。merged_core の約 2.8 倍の枚数 × 30/80 周）。
学習中のログは 1 周ごとに 1〜2 行ずつ増えます。`mAP50` の列が少しずつ上がっていけば順調です。

> ⚠️ **接続が切れたら、この下の学習のセルの代わりに、セクション 5-2 の「再開のセル」を実行してください。**
> このセルをもう一度実行すると、最初から学習し直しになります（前の途中経過は使われず、`…2` のような新しいフォルダができます）。

In [ ]:
from ultralytics import YOLO

# 始め方ごとの設定（どちらも、拡張は v2 と同じ）
if START_FROM == 'v2':
    model = YOLO(V2_BEST)  # v2 の重みから続ける
    schedule = dict(epochs=30, patience=10, close_mosaic=5,
                    optimizer='SGD', lr0=0.001, lrf=0.1, warmup_epochs=1)  # 小さめの学習率
else:
    model = YOLO(MODEL_NAME)  # 事前学習済みの重みを読み込む（初回は自動でダウンロードされる）
    schedule = dict(epochs=80, patience=25, close_mosaic=15)

results = model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    batch=16,
    flipud=0.5,             # 上下反転
    degrees=15,             # 回転（±15°）
    scale=0.9,              # 拡大・縮小（0.1〜1.9 倍）。葉の大きさで決める癖を弱める
    translate=0.2,          # 位置のずらし（±20%）
    project=str(RUNS_DIR),  # 結果をドライブに保存する
    name=RUN_NAME,          # 前の学習を上書きしない名前（同じ名前があると末尾に番号が付く）
    **schedule,
)

RUN_DIR = Path(model.trainer.save_dir)  # 今回の結果フォルダ
print('結果の保存先:', RUN_DIR)

### 5-2. 途中で接続が切れたとき（再開のセル）

学習の途中経過（`last.pt`）は、ドライブの `runs/<RUN_NAME>/weights/` に 1 周ごとに保存されています。
接続が切れても、そこから続きを学習できます。

**使い方**
1. 「ランタイム」→「ランタイムのタイプを変更」で、GPU になっているか確かめる
2. **セクション 1〜4 のセルを、上から順に実行し直す**（ライブラリ・ドライブ・学習データの準備をやり直すため。`RUN_TAG` と `DATASET_ZIP` は前と同じにしておく）
3. セクション 5 の学習のセルは**飛ばして**、下の再開のセルを実行する

途中経過が見つからないときや、学習がすでに最後まで終わっているときは、そう表示されるだけで、何も起きません。
「すべてのセルを実行」で、学習の直後にこのセルが動いた場合も、何もしません。

In [ ]:
from ultralytics import YOLO

if 'results' in globals():
    # この接続の中で、すでに学習のセルが最後まで終わっている（「すべてのセルを実行」したときなど）
    print('この接続の中で学習が終わっているので、再開は不要です。')
else:
    RUN_DIR = latest_run()  # runs/ の中で、今回の実行名のいちばん新しいフォルダ
    last_pt = RUN_DIR / 'weights' / 'last.pt' if RUN_DIR else None
    if last_pt is None or not last_pt.exists():
        print(f'再開できる途中経過（runs/{RUN_NAME}*/weights/last.pt）が見つかりません。')
        print('RUN_TAG と DATASET_ZIP が前と同じか確かめてください。初めて学習するときは、セクション 5 の学習のセルを実行します。')
    else:
        print('再開する学習:', RUN_DIR)
        model = YOLO(last_pt)  # 最後に保存された途中の重み
        try:
            results = model.train(resume=True)  # 前回の設定（epochs・拡張など）を引き継いで、続きから学習する
        except AssertionError as e:
            # 学習がすでに最後まで終わっていると、Ultralytics はこのエラーを出す
            print('この学習は、すでに最後まで終わっています。セクション 6 に進んでください。')
            print('（詳しい内容:', e, '）')
            results = None
        RUN_DIR = Path(model.trainer.save_dir) if results is not None else RUN_DIR
        print('結果の保存先:', RUN_DIR)

## 6. 学習の経過を確認する

学習の結果フォルダにある `results.png` を表示します。

**読み方**
- `train/box_loss` などの **loss（損失）** は、下がっていれば学習が進んでいる
- `val/...loss` が途中から**上がり始めたら**、学習データを丸暗記し始めた（過学習）サイン
- `metrics/mAP50(B)` は、上がっていくほど良い。横ばいになったら、それ以上は伸びにくい

In [ ]:
from IPython.display import Image as ShowImage, display

# 接続が切れた後でも動くように、RUN_DIR がなければ最新の結果を探す
if 'RUN_DIR' not in globals():
    RUN_DIR = latest_run()
print('結果フォルダ:', RUN_DIR)

display(ShowImage(filename=str(RUN_DIR / 'results.png'), width=1000))

## 7. クラスごとの成績と混同行列

いちばん成績の良かった重み **`best.pt`** を使って、valid の画像で成績を測り直します。

**数字の読み方**（どれも 0〜1。1 に近いほど良い）

| 指標 | 意味 | 例 |
|---|---|---|
| 適合率（Precision） | 「見つけた」と言ったもののうち、本当に正しかった割合 | 0.9 → 見つけたものの 9 割は本物 |
| 再現率（Recall） | 本当にあるもののうち、見つけられた割合 | 0.8 → 10 個のうち 8 個は見つけた |
| mAP50 | 枠の位置もそこそこ合っているかを含めた総合点（枠の重なり 50% 以上を正解とする） | 0.7 以上なら、まずまず |
| mAP50-95 | 枠の位置の正確さをより厳しく見た総合点 | mAP50 より低く出るのが普通 |

In [ ]:
import pandas as pd

best_model = YOLO(RUN_DIR / 'weights' / 'best.pt')

# valid で評価する（グラフは runs/<結果フォルダ名>_val に保存される）
metrics = best_model.val(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    batch=16,
    project=str(RUNS_DIR),
    name=RUN_DIR.name + '_val',
    plots=True,
)

# valid の正解の枠の数（クラスごと）
gt_boxes = Counter()
for label in (DATA_DIR / 'valid' / 'labels').glob('*.txt'):
    for line in label.read_text().splitlines():
        if line.strip():
            gt_boxes[int(line.split()[0])] += 1

def class_scores(m):
    """評価結果から、クラスごと（と全体）の 適合率・再現率・mAP50・mAP50-95 を取り出す。"""
    scores = {}
    for k, class_id in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(k)
        scores[CLASS_NAMES[int(class_id)]] = {'適合率 (P)': p, '再現率 (R)': r, 'mAP50': ap50, 'mAP50-95': ap}
    scores['全体'] = {'適合率 (P)': m.box.mp, '再現率 (R)': m.box.mr, 'mAP50': m.box.map50, 'mAP50-95': m.box.map}
    return scores

# クラスごとの表を作る
new_scores = class_scores(metrics)
name_to_id = {name: i for i, name in CLASS_NAMES.items()}
rows = []
for name, s in new_scores.items():
    count = sum(gt_boxes.values()) if name == '全体' else gt_boxes[name_to_id[name]]
    rows.append({'クラス': name, '正解の枠数': count, **s})

pd.DataFrame(rows).set_index('クラス').style.format('{:.3f}', subset=['適合率 (P)', '再現率 (R)', 'mAP50', 'mAP50-95'])

### 7-2. 混同行列

「正解のクラス」と「モデルの答え」の組み合わせを数えた表です。

**読み方**
- 縦が **モデルの答え（予測）**、横が **正解** です
- 対角線（左上→右下）に数が集まっているほど良い
- `background` は「何もない」という意味です
  - 横が `four_leaf` で縦が `background` のマス → 四つ葉を**見逃した**数
  - 縦が `four_leaf` で横が `background` のマス → 何もない所を四つ葉と**間違えた**数
  - `four_leaf` と `three_leaf` が入れ替わっているマス → 四つ葉と三つ葉を**取り違えた**数
- 正規化版（normalized）は、列ごとに割合にしたものです

In [ ]:
# 数で見る（表）
cm = metrics.confusion_matrix.matrix  # 行 = 予測、列 = 正解（最後が background）
labels = [CLASS_NAMES[i] for i in sorted(CLASS_NAMES)] + ['background']
display(pd.DataFrame(cm.astype(int),
                     index=[f'予測: {l}' for l in labels],
                     columns=[f'正解: {l}' for l in labels]))

# 図で見る
VAL_DIR = Path(metrics.save_dir)
display(ShowImage(filename=str(VAL_DIR / 'confusion_matrix.png'), width=600))
display(ShowImage(filename=str(VAL_DIR / 'confusion_matrix_normalized.png'), width=600))

### 7-3. 前のモデルと比べる

前のモデル（`PREV_BEST`）と今回のモデルを、**同じ valid の画像**で評価して、並べて比べます。

**読み方**
- 「差」は **今回 − 前** です。プラスなら今回のほうが良く、マイナスなら前のほうが良い
- 今回の目的は「三つ葉を四つ葉と間違える」のを減らすことなので、特に次の 2 つを見ます
  - **four_leaf の適合率**：上がっていれば、「四つ葉」と言ったものに三つ葉が混じりにくくなった
  - **三つ葉を四つ葉と間違えた数**（混同行列の「予測: four_leaf / 正解: three_leaf」のマス）：減っていれば良い
- 再現率が大きく下がっていたら、四つ葉の見逃しが増えています。適合率とのバランスで判断します
- **注意: 前のモデルには、少し有利な比べ方です。** 今回の valid の写真の一部は、前のモデルを学習したときの train に入っていました（データの分け方を作り直したため）。前のモデルは「見たことのある写真」で試験されることになるので、前のモデルの数字は実際より良く出ます。それでも今回のモデルが同じか上なら、良くなったと考えてよいです
- 差が小さいときは、自分の写真（`predict_photos.ipynb`）でも確かめてください

**merged_hardneg2・merged_hardneg2b のときの、採用の目安**（v2 と比べて）

必須（1 つでも外れたら採用しない）
- valid の **four_leaf の mAP50** が、v2 より 0.01 以上下がらない（下のセルで表示。`PREV_BEST` は v2）
- valid の **three_leaf の mAP50** が、v2 より 0.03 以上下がらない（下のセルで表示）
- 三つ葉だけの写真 100 枚（three-leaves）の、四つ葉の誤検出（確かめる前）が **63 個以下**（v3 と同じか、それより少ない）
- 代わりの写真（three-leaves の IMG_1403）の、四つ葉の誤検出（確かめる前）が **12 本より少ない**（v3 は 12 本）

「確かめる」を使うための目安（届かなければ、「確かめる」を機能ごと外すことを相談する）
- 確かめたあと、本物の四つ葉が **90% 以上**残る
- 確かめたあと、three-leaves の四つ葉の誤検出が **30 個以下**

three-leaves と本物の四つ葉の数は、パソコンで測る（`python training/eval_hardneg.py`。training/README.md）

In [ ]:
if not PREV_BEST.exists():
    print(f'前のモデル {PREV_BEST} が見つからないため、比較を飛ばします（セクション 3 の PREV_BEST を直してください）')
else:
    # 前のモデルを、同じ valid・同じ画像サイズで評価する（グラフは runs/<今回の結果フォルダ名>_val_prev に保存）
    prev_metrics = YOLO(PREV_BEST).val(
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        batch=16,
        project=str(RUNS_DIR),
        name=RUN_DIR.name + '_val_prev',
        plots=True,  # 混同行列を作るために必要
    )
    prev_scores = class_scores(prev_metrics)

    # クラスごとに、前・今回・差 を並べる
    rows = []
    for name in new_scores:
        row = {'クラス': name}
        for key in ['適合率 (P)', '再現率 (R)', 'mAP50']:
            before, after = prev_scores.get(name, {}).get(key, float('nan')), new_scores[name][key]
            row[f'{key} 前'] = before
            row[f'{key} 今回'] = after
            row[f'{key} 差'] = after - before
        rows.append(row)
    compare = pd.DataFrame(rows).set_index('クラス')
    display(compare.style.format('{:+.3f}', subset=[c for c in compare.columns if c.endswith('差')])
                         .format('{:.3f}', subset=[c for c in compare.columns if not c.endswith('差')]))

    # 三つ葉を四つ葉と間違えた数（混同行列: 行 = 予測 four_leaf(0)、列 = 正解 three_leaf(1)）
    mistaken_before = int(prev_metrics.confusion_matrix.matrix[0, 1])
    mistaken_after = int(metrics.confusion_matrix.matrix[0, 1])
    print(f'三つ葉を四つ葉と間違えた数: 前 {mistaken_before} → 今回 {mistaken_after}')

    # 採用の目安（valid の分）: four_leaf の mAP50 が 0.01 以上、three_leaf の mAP50 が 0.03 以上、下がらない
    drop4 = prev_scores['four_leaf']['mAP50'] - new_scores['four_leaf']['mAP50']
    drop3 = prev_scores['three_leaf']['mAP50'] - new_scores['three_leaf']['mAP50']
    print(f"four_leaf の mAP50 の下がり: {drop4:+.3f}（0.01 未満なら合格）→ {'合格' if drop4 < 0.01 else '不合格'}")
    print(f"three_leaf の mAP50 の下がり: {drop3:+.3f}（0.03 未満なら合格）→ {'合格' if drop3 < 0.03 else '不合格'}")
    print('三つ葉だけの写真 100 枚の誤検出（206 個以下で合格）は、パソコンで測ります（training/README.md）')

## 8. valid の画像 10 枚で、検出結果を目で確認する

数字だけでなく、実際の写真で「どこを見つけて、どこを見逃したか」を確かめます。
枠の上の数字は、モデルの自信の度合いです（ここは確認用で、アプリには表示しません）。

**見るポイント**
- 四つ葉に枠が付いているか（見逃しはないか）
- 三つ葉を四つ葉と間違えていないか
- 葉ではない所に枠が出ていないか

In [ ]:
import random
import matplotlib.pyplot as plt

random.seed(0)  # 毎回同じ 10 枚を選ぶ
valid_images = sorted((DATA_DIR / 'valid' / 'images').iterdir())
samples = random.sample(valid_images, min(10, len(valid_images)))

predictions = best_model.predict([str(p) for p in samples], imgsz=IMGSZ, conf=0.25, verbose=False)

fig, axes = plt.subplots(5, 2, figsize=(14, 32))
for ax, pred, path in zip(axes.flat, predictions, samples):
    ax.imshow(pred.plot()[:, :, ::-1])  # plot() は BGR なので RGB に並べ替える
    ax.set_title(path.name[:40], fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 9. ONNX に書き出す（アプリ用）

ブラウザで動かせる **ONNX** 形式に変換します。

| 設定 | 値 | 意味 |
|---|---|---|
| `imgsz` | 960 | アプリでも 960×960 の画像を入力する |
| `opset` | 12 | ONNX の版。ブラウザ用の実行環境で動きやすい版を選んでいる |
| `simplify` | True | 計算の手順を整理して、ファイルを軽く・速くする |
| `dynamic` | False | 入力サイズを 960×960 に固定する |

書き出したファイルは、ドライブの `MyDrive/yotsuba/` に **`ONNX_NAME`（例: `yotsuba_hardneg_hardneg2b.onnx`）** という名前で保存し、パソコンにもダウンロードします。
前の `yotsuba.onnx` は上書きしません。

In [ ]:
# best.pt を ONNX に変換する
onnx_path = best_model.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True, dynamic=False)
print('書き出したファイル:', onnx_path)

# ドライブに、実行名を含んだ名前（ONNX_NAME）で保存する（前の yotsuba.onnx は上書きしない）
ONNX_DRIVE_PATH = DRIVE_DIR / ONNX_NAME
shutil.copy(onnx_path, ONNX_DRIVE_PATH)
print(f'ドライブに保存しました: {ONNX_DRIVE_PATH}（{ONNX_DRIVE_PATH.stat().st_size / 1e6:.1f} MB）')

In [ ]:
import numpy as np
import onnxruntime as ort

# 書き出したモデルを実際に動かして、入力と出力の形（shape）を確かめる
session = ort.InferenceSession(str(ONNX_DRIVE_PATH), providers=['CPUExecutionProvider'])

for i in session.get_inputs():
    print(f'入力: {i.name}  shape={i.shape}  型={i.type}')

dummy = np.zeros((1, 3, IMGSZ, IMGSZ), dtype=np.float32)  # 真っ黒な画像を 1 枚入れてみる
outputs = session.run(None, {session.get_inputs()[0].name: dummy})
for o, value in zip(session.get_outputs(), outputs):
    print(f'出力: {o.name}  shape={value.shape}')

**出力の形の読み方**（例: `(1, 6, 18900)`）

- `1` … 一度に入れた画像の枚数
- `6` … 1 つの候補ごとの数字の個数 = 枠の位置 4 つ（中心 x・中心 y・幅・高さ、ピクセル単位）＋ クラスごとのスコア 2 つ（four_leaf、three_leaf）
- `18900` … 候補の数（960px の画像を 8・16・32px のマス目に分けた数: 120² + 60² + 30²）

アプリ側では、この 18900 個の候補からスコアの高いものを選び、重なった枠を 1 つにまとめる処理（NMS）を行います。

In [ ]:
from google.colab import files

# パソコンにダウンロードする（ブラウザのダウンロード確認が出たら許可する）
files.download(str(ONNX_DRIVE_PATH))

## 10. おわりに

| できたもの | 場所 |
|---|---|
| 学習済みの重み | `MyDrive/yotsuba/runs/<RUN_NAME>/weights/best.pt` |
| 学習のグラフ | `MyDrive/yotsuba/runs/<RUN_NAME>/results.png` |
| 評価のグラフ | `MyDrive/yotsuba/runs/<RUN_NAME>_val/`（前のモデルは `_val_prev`） |
| アプリ用モデル | `MyDrive/yotsuba/yotsuba_<RUN_TAG>.onnx` |

**アプリに入れるとき**（セクション 7-3 で、前のモデルより良いと確かめてから）
1. ダウンロードした ONNX を、**版を上げた名前**にして `public/models/` に置く（例: `yotsuba-v2.onnx`）
2. `src/lib/detector-config.ts` の `MODEL_URL` を、その名前に直す

同じ名前のまま中身だけ入れ替えると、ブラウザやサービスワーカーに残った古いモデルが使われ続けるので、必ず名前を変えてください。